In [13]:
!pip install openpyxl

In [14]:
import requests
import pandas as pd
import numpy as np

from bs4 import BeautifulSoup  # Vajalik peidetud väljade lugemiseks (pip install beautifulsoup4)
import io

In [15]:
# 1. ADDED 'file_path' HERE so the function can accept the filename argument
def import_attached_real_estate_data(file_path):
    # 2. CHANGED TO pd.read_excel because your file is an .xlsx spreadsheet
    # Added engine='openpyxl' to ensure seamless mnemomic cell mapping
    df = pd.read_excel(file_path, skiprows=4, engine="openpyxl")

    # Trim any hidden trailing/leading whitespaces from header names
    df.columns = df.columns.str.strip()

    # The file combines Quarters and Counties in a hierarchical view.
    # We locate the actual Quarter labels (e.g., '1996 I') and push them down.
    df["Kvartal_Clean"] = df.iloc[:, 0].where(
        df.iloc[:, 0].astype(str).str.contains(r"\d{4}", na=False)
    )
    df["Kvartal_Clean"] = df["Kvartal_Clean"].ffill()

    # 3. Clean and filter structural layout artifacts
    # Strip total calculation summary rows ('KOKKU') and completely empty lines
    df = df[df["Nimi"] != "KOKKU"]
    df = df.dropna(subset=["Nimi"])

    # 4. Standardize analytical metrics and filter baseline column metrics
    df = df[
        [
            "Kvartal_Clean",
            "Nimi",
            "Arv",
            "Kogupind (ha)",
            "Koguväärtus (eur)",
        ]
    ]
    df.columns = [
        "quarter",
        "county",
        "transaction_count",
        "total_area_ha",
        "total_value_eur",
    ]

    # 5. Clean text characters and enforce numeric variable assignments
    # '***' indicators are legally bound to at least <5 data rows and act as NaN targets
    for col in ["transaction_count", "total_area_ha", "total_value_eur"]:
        df[col] = df[col].astype(str).str.replace(" ", "").str.strip()
        df[col] = df[col].replace("***", np.nan)
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Määrame täisarvulistele veergudele täpse täisarvu tüübi, 
    # mis säilitab suured arvud ja lubab samal ajal ka tühje (NaN) lahtreid.
    df["transaction_count"] = df["transaction_count"].astype("Int64")
    df["total_value_eur"] = df["total_value_eur"].astype("Int64")
    # Pindalale jätame ujukoma, kuna seal on reaalsed komakohad, 
    # aga float64 on väikeste arvude puhul piisavalt täpne.
    df["total_area_ha"] = df["total_area_ha"].astype("float64")


    # Re-align raw index pointer counters
    df = df.reset_index(drop=True)

    return df  # <--- SEE rida lõpetab funktsiooni!

# === EXECUTION EXEC ===
# This will now successfully open the Excel sheet and pass it into the function parameters
df_real_estate = import_attached_real_estate_data(
    "Kinnisvara hinnastatistika maakondade loikes_raw.xlsx"
)

# View the clean structural output matrix frame
print(df_real_estate.head(15))


     quarter              county  transaction_count  total_area_ha  \
0     1996 I       Harju maakond                  1            NaN   
1     1996 I       Tartu maakond                  3            NaN   
2    1996 II       Harju maakond                  8            3.0   
3    1996 II  Lääne-Viru maakond                  2            NaN   
4    1996 II       Rapla maakond                  2            NaN   
5    1996 II       Tartu maakond                  3            NaN   
6    1996 II        Võru maakond                  1            NaN   
7   1996 III       Harju maakond                 55           54.0   
8   1996 III    Ida-Viru maakond                  1            NaN   
9   1996 III  Lääne-Viru maakond                  2            NaN   
10  1996 III       Põlva maakond                  2            NaN   
11  1996 III       Tartu maakond                  3            NaN   
12   1996 IV       Harju maakond                100           79.0   
13   1996 IV      Jõ

In [16]:
df_real_estate.to_csv("maa_amet_py_final.csv", index=False)

PermissionError: [Errno 13] Permission denied: 'maa_amet_py_final.csv'